# Document Categorization — EDA, Model Development and Evaluation

This notebook documents the data, model-development path and release evidence for the multilingual document categorization pipeline.

Current release: **microsoft/mdeberta-v3-base** classification + English/Spanish spaCy NER/tagging + MarianMT EN→ES augmentation + temperature-scaled confidence + XLA inference.

Development analysis uses **train + validation only**. The notebook never re-runs the held-out final evaluator and never reads `test.csv`; final results are loaded from frozen reports.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
REPORTS = ROOT / "reports"
DATA = ROOT / "data" / "processed_data"

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

dataset_summary = load_json(REPORTS / "revision2" / "dataset_summary.json")
preflight = load_json(REPORTS / "revision2" / "preflight.json")
checkpoint_selection = load_json(REPORTS / "revision2" / "checkpoint_selection.json")
calibration = load_json(REPORTS / "revision2" / "calibration.json")
production_validation = load_json(REPORTS / "revision2" / "production_validation_verification.json")
revision1_metrics = load_json(REPORTS / "revision1_performance_metrics.json")
revision2_metrics = load_json(REPORTS / "revision2" / "performance_metrics.json")
print("Frozen evidence loaded; test.csv read: False")


## 1. Dataset and development splits

The release corpus is based on **20 Newsgroups**. Retained English documents are mirrored into Spanish with MarianMT while each EN/ES pair remains in one split.

Frozen Revision 2 corpus: **11,861 source documents / 23,722 EN+ES rows / 12 categories / 2 languages**.

Validation is deterministic and whole-thread grouped. A planned temporal split was abandoned before retraining because parseable `Date:` coverage in official train was zero; the change is documented in `docs/revision2_amendment_01.md`.


In [ ]:
snapshot = pd.Series({
    "documents": dataset_summary["documents"],
    "source_documents": dataset_summary["source_documents"],
    "categories": dataset_summary["categories"],
    "languages": ", ".join(dataset_summary["languages"]),
    "train_documents": preflight["design_splits"]["train_documents"],
    "validation_documents": preflight["design_splits"]["validation_documents"],
    "train_source_pairs": preflight["design_splits"]["train_source_pairs"],
    "validation_source_pairs": preflight["design_splits"]["validation_source_pairs"],
    "train_validation_exact_text_overlap": preflight["design_splits"]["train_validation_exact_text_overlap"],
})
display(snapshot.to_frame("Revision 2"))

paths = {"train": DATA / "train.csv", "validation": DATA / "validation.csv"}
missing = [str(p) for p in paths.values() if not p.exists()]
if missing:
    raise FileNotFoundError("Run `python scripts/prepare_data.py` first: " + ", ".join(missing))
dev = {name: pd.read_csv(path).reset_index(drop=True) for name, path in paths.items()}
for name, frame in dev.items():
    if "split" in frame:
        assert frame["split"].astype(str).eq(name).all()
train, validation = dev["train"], dev["validation"]
print({name: len(frame) for name, frame in dev.items()})
print("test.csv read: False")


## 2. EDA — balance, quality and text length

In [ ]:
category_counts = pd.concat(
    [frame["label"].value_counts().rename(name) for name, frame in dev.items()], axis=1
).fillna(0).astype(int)
display(category_counts)
category_counts.plot(kind="bar", figsize=(13, 5), title="Category distribution — development splits")
plt.ylabel("documents"); plt.tight_layout(); plt.show()

language_counts = pd.concat(
    [frame["language"].value_counts().rename(name) for name, frame in dev.items()], axis=1
).fillna(0).astype(int)
display(language_counts)

dev_data = pd.concat([frame.assign(split=name) for name, frame in dev.items()], ignore_index=True)
coverage = pd.crosstab(dev_data["label"], dev_data["language"])
display(coverage)
assert {"en", "es"}.issubset(set(dev_data["language"].astype(str)))
assert (coverage > 0).all().all()


In [ ]:
quality = pd.DataFrame({
    "missing_text": [int(frame["text"].isna().sum()) for frame in dev.values()],
    "empty_text": [int(frame["text"].fillna("").str.strip().eq("").sum()) for frame in dev.values()],
    "within_split_exact_duplicates": [int(frame["text"].duplicated().sum()) for frame in dev.values()],
}, index=dev.keys())
display(quality)
assert quality[["missing_text", "empty_text"]].to_numpy().sum() == 0
assert preflight["design_splits"]["train_validation_exact_text_overlap"] == 0

for name, frame in dev.items():
    if {"pair_id", "language"}.issubset(frame.columns):
        assert frame.groupby("pair_id")["language"].nunique().eq(2).all(), name

lengths = dev_data[["split", "language", "text"]].copy()
lengths["words"] = lengths["text"].astype(str).str.split().str.len()
lengths["characters"] = lengths["text"].astype(str).str.len()
display(lengths.groupby(["split", "language"])[["words", "characters"]].describe(percentiles=[.5,.9,.95,.99]))

for language, group in lengths.groupby("language"):
    group["words"].clip(upper=group["words"].quantile(.99)).hist(alpha=.5, bins=50, label=language)
plt.legend(); plt.title("Development word counts — clipped at p99"); plt.xlabel("words"); plt.show()


In [ ]:
display(
    dev_data.groupby(["split", "language", "label"], group_keys=False)
    .head(1)[["split", "language", "label", "text"]]
    .head(24)
)


## 3. Revision 2 preprocessing

Model representation:

```text
Subject without repeated leading Re:

Body
```

Routing/sender metadata is dropped. `sklearn` removes footers and quoted reply content; structural and token-density filters run next. Classification uses a deterministic **150-word** source window. Spanish is translated from the cleaned English representation and normalized again. Cross-split exact-text leakage is removed by dropping whole EN/ES pairs.

Repeated `Re:` markers were removed because a train-only diagnostic showed strong category association above the pre-registered threshold; they were treated as an accidental cue rather than useful document content.


In [ ]:
display(pd.DataFrame(preflight["token_budget"]["rows"]))
display(pd.Series(preflight["translation"], name="translation sanity").to_frame())
print("Model:", preflight["token_budget"]["model_name"])
print("Classification source window: 150 words")
print("Test metrics used for model selection:", preflight["model_selection_metrics_from_test"])


## 4. Baseline and transfer learning

Baseline: **TF-IDF word unigrams + Logistic Regression** on the same Revision 2 development data.

Current classifier: `microsoft/mdeberta-v3-base`, fine-tuned with TensorFlow/Keras for 5 epochs.

Frozen training configuration: LR `2e-5`, batch size `2`, max model tokens `512`, AdamW, weight decay `0.01`, 10% warmup, global gradient clip `1.0`.

Checkpoint selection is deterministic: highest validation correct-document count → lower validation loss → earlier epoch. Revision 2 selected **epoch 5: 1,886 / 2,186 correct (86.28%)**.


In [ ]:
baseline = pd.Series({
    "validation_accuracy": preflight["baseline_validation"]["accuracy"],
    "validation_macro_f1": preflight["baseline_validation"]["macro_f1"],
    "english_accuracy": preflight["baseline_validation"]["per_language"]["en"]["accuracy"],
    "spanish_accuracy": preflight["baseline_validation"]["per_language"]["es"]["accuracy"],
}, name="TF-IDF + Logistic Regression")
display(baseline.to_frame())

history = pd.read_csv(REPORTS / "revision2" / "training_history.csv")
history["epoch"] = history["epoch"] + 1
display(history)
display(pd.Series(checkpoint_selection, name="selected checkpoint").to_frame())

history.plot(x="epoch", y=["loss", "val_loss"], marker="o", title="Fine-tuning loss")
plt.ylabel("loss"); plt.show()
history.plot(x="epoch", y=["accuracy", "val_accuracy"], marker="o", title="Fine-tuning accuracy")
plt.ylabel("accuracy"); plt.show()


Training accuracy reaches ~98.7% while validation accuracy finishes at ~86.3%, so the run has a clear generalization gap. Validation loss is non-monotonic and reaches its minimum before the selected epoch. The selected checkpoint follows the frozen correct-document rule; validation loss remains a monitored diagnostic and tie-breaker.


## 5. Calibration and frozen runtime

The selected classifier uses scalar temperature scaling fitted on **validation only**. Calibration preserves argmax and changes confidence estimates, not predicted categories.

Release runtime: float32 + XLA, token buckets `64/128/192/256/384/512`, attention-balanced classifier batch sizes `32/32/16/16/4/4`, 150-word classification window, 75-word spaCy tagging window, and parallel classifier/tagger stages.


In [ ]:
calibration_summary = pd.DataFrame({
    "before": {
        "accuracy": calibration["before"]["accuracy"],
        "mean_confidence": calibration["before"]["mean_confidence"],
        "NLL": calibration["before"]["nll"],
        "ECE": calibration["before"]["ece"],
        "Brier": calibration["before"]["brier"],
    },
    "after": {
        "accuracy": calibration["after"]["accuracy"],
        "mean_confidence": calibration["after"]["mean_confidence"],
        "NLL": calibration["after"]["nll"],
        "ECE": calibration["after"]["ece"],
        "Brier": calibration["after"]["brier"],
    },
})
display(calibration_summary)
print("Temperature:", calibration["temperature"], "| argmax unchanged:", calibration["argmax_unchanged"])

runtime = pd.Series({
    "validation_accuracy": production_validation["accuracy"],
    "validation_macro_f1": production_validation["macro_f1"],
    "mean_calibrated_confidence": production_validation["mean_calibrated_confidence"],
    "end_to_end_docs_per_sec": production_validation["processing_speed_docs_per_sec"],
    "english_accuracy": production_validation["per_language"]["en"]["accuracy"],
    "spanish_accuracy": production_validation["per_language"]["es"]["accuracy"],
    "selected_correct": production_validation["selection_verification"]["selected_correct_documents"],
    "observed_correct": production_validation["selection_verification"]["observed_correct_documents"],
    "exact_correct_count_match": production_validation["selection_verification"]["exact_correct_count_match"],
}, name="frozen validation runtime")
display(runtime.to_frame())


## 6. Recorded release evaluation

This section reads immutable report files only; it does **not** run the final evaluator or open `test.csv`.

Revision 2 is the current release. Revision 1 and Revision 2 are development history rather than a controlled A/B test because Revision 2 changed representation, category selection and validation construction before retraining. The controlled comparison inside each revision is the transformer versus its baseline on the same data.


In [ ]:
comparison = pd.DataFrame({
    "Revision 1": {
        "test_accuracy": revision1_metrics["classification_accuracy"],
        "macro_f1": revision1_metrics["f1_score_macro"],
        "end_to_end_docs_per_sec": revision1_metrics["processing_speed_docs_per_sec"],
        "english_accuracy": revision1_metrics["per_language_accuracy"]["en"],
        "spanish_accuracy": revision1_metrics["per_language_accuracy"]["es"],
        "baseline_accuracy": revision1_metrics["baseline_accuracy"],
        "relative_improvement_over_baseline": revision1_metrics["accuracy_improvement_over_baseline_relative"],
        "mean_calibrated_confidence": revision1_metrics["mean_calibrated_confidence"],
    },
    "Revision 2": {
        "test_accuracy": revision2_metrics["classification_accuracy"],
        "macro_f1": revision2_metrics["f1_score_macro"],
        "end_to_end_docs_per_sec": revision2_metrics["processing_speed_docs_per_sec"],
        "english_accuracy": revision2_metrics["per_language_accuracy"]["en"],
        "spanish_accuracy": revision2_metrics["per_language_accuracy"]["es"],
        "baseline_accuracy": revision2_metrics["baseline_accuracy"],
        "relative_improvement_over_baseline": revision2_metrics["accuracy_improvement_over_baseline_relative"],
        "mean_calibrated_confidence": revision2_metrics["mean_calibrated_confidence"],
    },
})
display(comparison)

print("Current release accuracy:", f"{revision2_metrics['classification_accuracy']:.2%}")
print("Current release macro F1:", f"{revision2_metrics['f1_score_macro']:.2%}")
print("Current release throughput:", f"{revision2_metrics['processing_speed_docs_per_sec']:.2f} docs/s")

mcnemar = pd.DataFrame(revision2_metrics["significance"]["mcnemar_exact_by_language"]).T
display(mcnemar)
display(pd.Series(revision2_metrics["significance"]["cluster_bootstrap"], name="pair-cluster bootstrap").to_frame())


## 7. Optional optimization and reproducibility

`utils/model_optimization.py` provides a TensorFlow Lite post-training quantization path. It is separate from the validated float32/XLA release path and should be benchmarked independently before replacing it.

Key workflow:

```text
scripts/prepare_data.py
scripts/preflight_revision2.py
scripts/train_revision2.py
scripts/calibrate_validation.py
scripts/freeze_production.py
scripts/verify_production_validation.py
scripts/evaluate.py                 # displays preserved final metrics only
app/real_time_dashboard.py
```

The historical one-time Revision 2 final evaluator remains for provenance, while the completed consumption marker prevents another final-test run.
